## UrbanRide_09 — Gold Payment Features
**Method:** Batch enrichment — Silver → Gold  
**Inputs:**
- `urbanride.silver.payments` — base payment data (19.6M rows)
- `urbanride.gold.trip_features` — trip context, already enriched with driver + customer attributes

**Output:** `urbanride.gold.payment_features`  
**Schedule:** Daily · runs after UrbanRide_08 (Gold Trip Features must exist first)  
**Partition:** `payment_status`  
**ZORDER:** `payment_date`, `customer_id`

**Grain:** one row per `payment_id`

**Feature engineering:**
- Direct payment attributes from silver.payments
- Derived: `payment_date`, `is_suspicious` (combined fraud flag)
- Trip context enrichment via join on trip_id: city, trip_date, distance, surge, driver rating

**Feeds:** Fraud Detection Model (12)

**Note:** Gold Trip Features must be built before this notebook runs.

In [0]:
from pyspark.sql.functions import (
    col, when, lit, current_timestamp,
    round, broadcast
)

CATALOG = 'urbanride'
SILVER  = f'{CATALOG}.silver'
GOLD    = f'{CATALOG}.gold'

spark.sql(f'USE CATALOG {CATALOG}')

# Dependency check — Gold Trip Features must exist before this notebook runs
if not spark.catalog.tableExists(f'{GOLD}.trip_features'):
    raise Exception(
        f'{GOLD}.trip_features does not exist. '
        'Run UrbanRide_08_Gold_Trip_Features first.'
    )

print(f'Catalog : {CATALOG}')
print(f'Sources : {SILVER}.payments, {GOLD}.trip_features')
print(f'Target  : {GOLD}.payment_features')
print('Dependency check passed — gold.trip_features exists.')
print('Config ready.')


Catalog : urbanride
Sources : urbanride.silver.payments, urbanride.gold.trip_features
Target  : urbanride.gold.payment_features
Dependency check passed — gold.trip_features exists.
Config ready.


In [0]:
# Gold payment_features uses incremental append — same as trip_features
# Reason: payment rows are immutable once written to Silver
# A payment that happened on Day 1 never changes — no recalculations needed
# Strategy: append only new payments processed since last Gold run
# Watermark: silver.payments._processed_at vs gold.payment_features._processed_at

print('Checking load type...')

table_exists = spark.catalog.tableExists(f'{GOLD}.payment_features')

if not table_exists:
    LOAD_TYPE = 'full'
    last_run  = None
    print('  Gold table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'incremental'
    last_run  = spark.sql(
        f'SELECT MAX(_processed_at) FROM {GOLD}.payment_features'
    ).collect()[0][0]
    print(f'  Gold table exists — INCREMENTAL LOAD')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')


Checking load type...
  Gold table does not exist — FULL LOAD
  Load type : full


In [0]:
print('Reading source tables...')

# Payments — filter by watermark for incremental
df_payments_full = spark.table(f'{SILVER}.payments')

if LOAD_TYPE == 'full':
    df_payments = df_payments_full
    print('  Full load — reading all payment rows')
else:
    df_payments = df_payments_full.filter(col('_processed_at') > last_run)
    print(f'  Incremental — reading payments processed after {last_run}')

payment_count = df_payments.count()
print(f'  Payments to process : {payment_count:,}')

if payment_count == 0:
    print('  No new payments to process — Gold already up to date.')
    dbutils.notebook.exit('No new rows — skipping')

# Gold trip features — select only columns needed for enrichment
# Filter completed trips only — cancelled trips have no valid payment context
df_trip_lookup = spark.table(f'{GOLD}.trip_features') \
    .filter(col('trip_status') == 'completed') \
    .select(
        col('trip_id'),
        col('city'),
        col('trip_date'),
        col('vehicle_type'),
        col('weather'),
        col('day_type'),
        col('distance_km'),
        col('fare_per_km'),
        col('surge_multiplier'),
        col('is_peak_surge'),
        col('driver_rating'),
        col('is_high_value_driver'),
        col('is_ghost_trip'),
        col('is_distance_invalid'),
        col('is_churned'),
    )

print(f'  Trip lookup rows    : {df_trip_lookup.count():,}')


Reading source tables...
  Full load — reading all payment rows
  Payments to process : 19,600,753
  Trip lookup rows    : 19,542,216


In [0]:
print('Building payment base features...')

df_payment_base = df_payments.select(
    col('payment_id'),
    col('trip_id'),
    col('customer_id'),
    col('amount'),
    col('payment_method'),
    col('payment_timestamp'),
    col('payment_status'),
    col('trip_fare_amount'),
    col('fare_delta_pct'),
    col('is_fraud_card'),
    col('is_orphan_payment'),
    col('is_fare_mismatch'),
    col('is_status_invalid'),
    col('is_method_invalid'),

    # Derived: payment_date — for partitioning and daily aggregations
    col('payment_timestamp').cast('date').alias('payment_date'),

    # Derived: is_suspicious — combined fraud signal
    # Any one of these flags being true makes the payment suspicious
    # Used as the label for the Fraud Detection model
    when(
        (col('is_fraud_card')     == True) |
        (col('is_fare_mismatch')  == True) |
        (col('is_orphan_payment') == True),
        True
    ).otherwise(False).alias('is_suspicious'),
)

print(f'  Base feature rows : {df_payment_base.count():,}')
print(f'  Columns so far    : {len(df_payment_base.columns)}')

# Profile suspicious payments before enrichment
suspicious_count = df_payment_base.filter(col('is_suspicious') == True).count()
pct = suspicious_count / df_payment_base.count() * 100
print(f'  Suspicious payments : {suspicious_count:,} ({pct:.2f}%)')


Building payment base features...
  Base feature rows : 19,600,753
  Columns so far    : 16
  Suspicious payments : 145,036 (0.74%)


In [0]:
print('Enriching payments with trip context...')

# Join strategy: sort merge join
# Both tables are large (19.6M payments, 19.5M completed trips)
# Broadcast not suitable — trip lookup is too large after filtering
# Left join — preserve all payments including orphans (no matching trip)

df_gold = df_payment_base.join(
    df_trip_lookup,
    on='trip_id',
    how='left'
)

# Add metadata
df_gold = df_gold \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit('silver.payments+gold.trip_features'))

# Verify orphan payments have null trip context as expected
orphan_null_check = df_gold \
    .filter(
        (col('is_orphan_payment') == True) &
        (col('city').isNull())
    ).count()

print(f'  Final row count   : {df_gold.count():,}')
print(f'  Final column count: {len(df_gold.columns)}')
print(f'  Orphan payments with null trip context : {orphan_null_check:,} (expected > 0)')
print()
print('Gold schema:')
df_gold.printSchema()


Enriching payments with trip context...
  Final row count   : 19,600,753
  Final column count: 32
  Orphan payments with null trip context : 58,537 (expected > 0)

Gold schema:
root
 |-- trip_id: string (nullable = true)
 |-- payment_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_timestamp: timestamp (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- trip_fare_amount: double (nullable = true)
 |-- fare_delta_pct: double (nullable = true)
 |-- is_fraud_card: boolean (nullable = true)
 |-- is_orphan_payment: boolean (nullable = true)
 |-- is_fare_mismatch: boolean (nullable = true)
 |-- is_status_invalid: boolean (nullable = true)
 |-- is_method_invalid: boolean (nullable = true)
 |-- payment_date: date (nullable = true)
 |-- is_suspicious: boolean (nullable = false)
 |-- city: string (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- vehi

In [0]:
import time
print(f'Writing gold.payment_features — mode: {LOAD_TYPE}...')
t0 = time.time()

if LOAD_TYPE == 'full':
    df_gold.write \
        .format('delta') \
        .mode('overwrite') \
        .partitionBy('payment_status') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{GOLD}.payment_features')
    print(f'  Full load written : {df_gold.count():,} rows')
    print(f'  Write time        : {time.time()-t0}s')
    print('Running OPTIMIZE + ZORDER...')
    spark.sql(f'OPTIMIZE {GOLD}.payment_features ZORDER BY (payment_date, customer_id)')
    print('  OPTIMIZE + ZORDER complete')
else:
    new_count = df_gold.count()
    if new_count > 0:
        df_gold.write \
            .format('delta') \
            .mode('append') \
            .saveAsTable(f'{GOLD}.payment_features')
        print(f'  Incremental rows appended : {new_count:,}')
        print(f'  Write time                : {time.time()-t0}s')
        print('Running OPTIMIZE + ZORDER...')
        spark.sql(f'OPTIMIZE {GOLD}.payment_features ZORDER BY (payment_date, customer_id)')
        print('  OPTIMIZE + ZORDER complete')
    else:
        print('  No new rows — skipping write and OPTIMIZE')


Writing gold.payment_features — mode: full...
  Full load written : 19,600,753 rows
  Write time        : 35.63273239135742s
Running OPTIMIZE + ZORDER...
  OPTIMIZE + ZORDER complete


In [0]:
print('=== GOLD PAYMENT FEATURES VERIFICATION ===')
print()

df_verify = spark.table(f'{GOLD}.payment_features')
total = df_verify.count()

print(f'  Total rows    : {total:,}')
print(f'  Total columns : {len(df_verify.columns)}')
print(f'  Load type     : {LOAD_TYPE}')
print()

# Data quality checks
print('Data quality checks:')
checks = [
    ('Null payment_id',       df_verify.filter(col('payment_id').isNull()).count()),
    ('Null customer_id',      df_verify.filter(col('customer_id').isNull()).count()),
    ('Null amount',           df_verify.filter(col('amount').isNull()).count()),
    ('Null payment_date',     df_verify.filter(col('payment_date').isNull()).count()),
    ('Null payment_status',   df_verify.filter(col('payment_status').isNull()).count()),
    ('Negative amount',       df_verify.filter(col('amount') < 0).count()),
    ('Null is_suspicious',    df_verify.filter(col('is_suspicious').isNull()).count()),
]

all_passed = True
for check, result in checks:
    status = 'PASS' if result == 0 else 'WARN'
    if result > 0: all_passed = False
    print(f'  {status}  {check:<30} : {result:,}')

print()
print('Payment status distribution:')
df_verify.groupBy('payment_status').count().orderBy('count', ascending=False).show()

print('Payment method distribution:')
df_verify.groupBy('payment_method').count().orderBy('count', ascending=False).show()

print('Suspicious payment breakdown:')
from pyspark.sql.functions import sum as spark_sum, avg
df_verify.agg(
    spark_sum(when(col('is_suspicious')    == True, 1).otherwise(0)).alias('total_suspicious'),
    spark_sum(when(col('is_fraud_card')    == True, 1).otherwise(0)).alias('fraud_card'),
    spark_sum(when(col('is_fare_mismatch') == True, 1).otherwise(0)).alias('fare_mismatch'),
    spark_sum(when(col('is_orphan_payment')== True, 1).otherwise(0)).alias('orphan_payment'),
).show(truncate=False)

print('Suspicious vs non-suspicious — avg feature comparison:')
df_verify.groupBy('is_suspicious').agg(
    round(avg('amount'), 2).alias('avg_amount'),
    round(avg('fare_delta_pct'), 4).alias('avg_fare_delta_pct'),
    round(avg('driver_rating'), 4).alias('avg_driver_rating'),
    round(avg('surge_multiplier'), 4).alias('avg_surge_multiplier'),
).orderBy('is_suspicious').show()

print('City distribution (non-orphan payments only):')
df_verify.filter(col('city').isNotNull()) \
    .groupBy('city').count().orderBy('count', ascending=False).show()

print('Sample suspicious rows:')
df_verify.filter(col('is_suspicious') == True).select(
    'payment_id', 'customer_id', 'amount', 'payment_method',
    'payment_status', 'is_fraud_card', 'is_fare_mismatch',
    'is_orphan_payment', 'is_suspicious', 'driver_rating'
).limit(5).show(truncate=False)

print()
detail = spark.sql(f'DESCRIBE DETAIL {GOLD}.payment_features').collect()[0]
print(f'  numFiles    : {detail["numFiles"]}')
print(f'  sizeInBytes : {detail["sizeInBytes"]:,}')

print()
if all_passed:
    print('All quality checks passed. Gold payment features ready.')
else:
    print('Some checks flagged — review WARN items above.')
print('Next: UrbanRide_10 — Gold Funnel Metrics')


=== GOLD PAYMENT FEATURES VERIFICATION ===

  Total rows    : 19,600,753
  Total columns : 32
  Load type     : full

Data quality checks:
  PASS  Null payment_id                : 0
  PASS  Null customer_id               : 0
  PASS  Null amount                    : 0
  PASS  Null payment_date              : 0
  PASS  Null payment_status            : 0
  PASS  Negative amount                : 0
  PASS  Null is_suspicious             : 0

Payment status distribution:
+--------------+--------+
|payment_status|   count|
+--------------+--------+
|       success|19015283|
|        failed|  585470|
+--------------+--------+

Payment method distribution:
+--------------+--------+
|payment_method|   count|
+--------------+--------+
|           UPI|10807836|
|   Credit Card| 3905308|
|        Wallet| 2932303|
|          Cash| 1955306|
+--------------+--------+

Suspicious payment breakdown:
+----------------+----------+-------------+--------------+
|total_suspicious|fraud_card|fare_mismatch|orp

### Suspicious vs Non-Suspicious Comparison:
| Feature | Non-Suspicious | Suspicious | Signal? |
|---|---|---|---|
| avg_amount | 143.76 | 158.27 | ✅ Moderate — suspicious payments 10% higher |
| avg_fare_delta_pct | 0.0 | 0.0795 | ✅ Strong — fare mismatch clearly visible |
| avg_driver_rating | 4.2494 | 4.248 | ❌ Weak — nearly identical |
| avg_surge_multiplier | 1.3131 | 1.3122 | ❌ Weak — nearly identical |

### Suspicious Payment Breakdown:

| Flag | Count | % of total | Note |
|---|---|---|---|
| total_suspicious | 145,036 | 0.74% | Heavily imbalanced — class weights needed in fraud model |
| fraud_card | 53,017 | 0.27% | Flagged card hash |
| fare_mismatch | 33,565 | 0.17% | Payment amount differs from trip fare by > 5% |
| orphan_payment | 58,537 | 0.30% | No matching completed trip found |


### Key Observations for Model Building:

1. **IMBALANCED DATASET** — only **0.74% suspicious payments** out of 19.6M total.
   Fraud model must use class weights or oversampling (SMOTE) during training.
   Accuracy is a misleading metric here — use Precision, Recall, F1, AUC-ROC instead.

2. **fare_delta_pct IS THE STRONGEST FRAUD SIGNAL** — 0.0 for clean vs 0.0795 for suspicious.
   _**Expect this to be the top feature by importance in the fraud model.**_

3. **amount IS A MODERATE SIGNA**L — suspicious payments average 10% higher (158 vs 144).
   Include as a feature but don't over-rely on it.

4. **driver_rating AND surge_multiplier ARE WEAK SIGNALS** — nearly identical across both groups.
   Monitor feature importance — may be candidates for removal if they add noise.

5. **ORPHAN PAYMENTS have NULL trip context columns (city, driver_rating etc).**
   Fraud model must handle nulls — use imputation or null-safe features.
   is_orphan_payment flag itself is a strong direct signal — keep it as a feature.

6. **FRAUD HAPPENS ON SUCCESSFUL PAYMENTS** — _`not failed ones`_.
   **`Filter training data to payment_status = success for the fraud model.`**
   Failed payments are a separate problem (payment gateway issues, not fraud).

7. **preferred_payment_method IN GOLD CUSTOMER FEATURES HAS ZERO VARIANCE** — all UPI.
   Drop this column before feeding into churn and segmentation models.
   Use payment_method from Gold Payment Features instead for fraud model.